# Deep Learning Models for Oil Recovery Factor Prediction
## Comparative Study: MLP · LSTM · CNN · Transformer

**Dataset:** Proxy5 — polymer flood reservoir simulation  
**Target:** `Oil_recovery_factor (%)`  
**Features:** 14 reservoir / fluid / operational parameters  
**Framework:** Keras (TensorFlow back-end)

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import keras
from keras import layers, Model, Input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')

# ── Shared hyper-parameters ──────────────────────────────────────────────────
BATCH_SIZE = 64
EPOCHS     = 200
LR         = 1e-3
PATIENCE   = 20
TEST_SIZE  = 0.15
VAL_SIZE   = 0.15

---
## 2. Data Loading & Exploration

In [ ]:
# Place Proxy5.csv in the same folder as this notebook, or adjust the path.
DATA_PATH = 'Proxy5.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='YlGnBu', axis=1)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
TARGET   = 'Oil_recovery_factor (%)'
FEATURES = [c for c in df.columns if c != TARGET]
print(f'Features ({len(FEATURES)}):', FEATURES)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Oil Recovery Factor — distribution')
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Count')

axes[1].boxplot(df[TARGET], vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Box plot')
axes[1].set_xlabel(TARGET)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Pre-processing

In [ ]:
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32).reshape(-1, 1)

# 70 / 15 / 15 split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train)
X_val_s   = x_scaler.transform(X_val)
X_test_s  = x_scaler.transform(X_test)

y_train_s = y_scaler.fit_transform(y_train)
y_val_s   = y_scaler.transform(y_val)
y_test_s  = y_scaler.transform(y_test)

N_FEATURES = X_train_s.shape[1]
print(f'Input dimension: {N_FEATURES}')

---
## 4. Training Utilities

In [ ]:
def get_callbacks(name):
    """Standard Keras callbacks: early stopping + LR reduction + best checkpoint."""
    return [
        EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=10,
            min_lr=1e-6, verbose=0),
    ]


def fit_model(model, name):
    """Compile, fit, and return the history object."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='mse',
        metrics=['mae'],
    )
    history = model.fit(
        X_train_s, y_train_s,
        validation_data=(X_val_s, y_val_s),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks(name),
        verbose=0,
    )
    # Print final epoch metrics
    ep  = len(history.history['loss'])
    trL = history.history['loss'][-1]
    vaL = history.history['val_loss'][-1]
    print(f'[{name}] stopped at epoch {ep} — train_loss={trL:.5f}  val_loss={vaL:.5f}')
    return history


def evaluate_model(model, name):
    """Predict on test set, inverse-scale, and compute metrics."""
    preds_s   = model.predict(X_test_s, verbose=0)
    preds_inv = y_scaler.inverse_transform(preds_s)
    trues_inv = y_scaler.inverse_transform(y_test_s)

    rmse = np.sqrt(mean_squared_error(trues_inv, preds_inv))
    mae  = mean_absolute_error(trues_inv, preds_inv)
    r2   = r2_score(trues_inv, preds_inv)
    mape = np.mean(
        np.abs((trues_inv - preds_inv) /
               np.where(trues_inv == 0, 1e-8, trues_inv))
    ) * 100

    metrics = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}
    print(f'[{name}] Test → RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return preds_inv, trues_inv, metrics

---
## 5. Model 1 — Multi-Layer Perceptron (MLP)

A fully-connected feed-forward network.  
Most common baseline DL regressor; often highly competitive on tabular data.

In [ ]:
def build_mlp(n_features, hidden=(256, 128, 64, 32), dropout=0.3):
    inputs = Input(shape=(n_features,), name='mlp_input')
    x = inputs
    for units in hidden:
        x = layers.Dense(units, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name='output')(x)
    return Model(inputs, outputs, name='MLP')


mlp_model = build_mlp(N_FEATURES)
mlp_model.summary()

In [ ]:
mlp_history = fit_model(mlp_model, 'MLP')
mlp_preds, mlp_trues, mlp_metrics = evaluate_model(mlp_model, 'MLP')

---
## 6. Model 2 — 1-D Convolutional Neural Network (CNN)

Treats the 14 features as a 1-D sequence. Three `Conv1D` blocks learn local
feature interactions efficiently via sliding-window filters.

In [ ]:
def build_cnn(n_features, dropout=0.3):
    # Input shape: (n_features,)  →  reshape to (n_features, 1) for Conv1D
    inputs = Input(shape=(n_features,), name='cnn_input')
    x = layers.Reshape((n_features, 1))(inputs)

    for filters in (32, 64, 128):
        x = layers.Conv1D(filters, kernel_size=3, padding='same',
                          kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout)(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name='output')(x)
    return Model(inputs, outputs, name='CNN_1D')


cnn_model = build_cnn(N_FEATURES)
cnn_model.summary()

In [ ]:
cnn_history = fit_model(cnn_model, 'CNN')
cnn_preds, cnn_trues, cnn_metrics = evaluate_model(cnn_model, 'CNN')

---
## 7. Model 3 — LSTM (Long Short-Term Memory)

Each feature is treated as one time-step of a univariate signal.
Two stacked LSTM layers learn ordered feature interactions via gating.

In [ ]:
def build_lstm(n_features, hidden=128, num_layers=2, dropout=0.3):
    inputs = Input(shape=(n_features,), name='lstm_input')
    # Each feature becomes a time-step: (batch, n_features, 1)
    x = layers.Reshape((n_features, 1))(inputs)

    for i in range(num_layers):
        return_seq = (i < num_layers - 1)   # all but last layer return sequences
        x = layers.LSTM(hidden, return_sequences=return_seq,
                        dropout=dropout, recurrent_dropout=0.0)(x)

    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name='output')(x)
    return Model(inputs, outputs, name='LSTM')


lstm_model = build_lstm(N_FEATURES)
lstm_model.summary()

In [ ]:
lstm_history = fit_model(lstm_model, 'LSTM')
lstm_preds, lstm_trues, lstm_metrics = evaluate_model(lstm_model, 'LSTM')

---
## 8. Model 4 — Tabular Transformer

Each scalar feature is embedded into a *d_model*-dimensional vector.
Multi-head self-attention then learns all pairwise feature relationships.
A learnable `[CLS]` token aggregates global context for the regression head.

In [ ]:
class CLSToken(layers.Layer):
    """Prepends a trainable [CLS] token to the feature sequence."""
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

    def build(self, input_shape):
        self.cls = self.add_weight(
            name='cls_token', shape=(1, 1, self.d_model),
            initializer='truncated_normal', trainable=True)
        super().build(input_shape)

    def call(self, x):
        batch = tf.shape(x)[0]
        cls   = tf.tile(self.cls, [batch, 1, 1])
        return tf.concat([cls, x], axis=1)


class TransformerEncoderBlock(layers.Layer):
    """One standard Transformer encoder block."""
    def __init__(self, d_model, nhead, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn  = layers.MultiHeadAttention(
            num_heads=nhead, key_dim=d_model // nhead, dropout=dropout)
        self.ffn   = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, training=False):
        attn_out = self.attn(x, x, training=training)
        x        = self.norm1(x + self.drop1(attn_out, training=training))
        ffn_out  = self.ffn(x, training=training)
        return self.norm2(x + self.drop2(ffn_out, training=training))


def build_transformer(n_features, d_model=64, nhead=4,
                       num_blocks=3, ff_dim=256, dropout=0.1):
    inputs = Input(shape=(n_features,), name='tfm_input')

    # Embed each feature: (batch, n_features) → (batch, n_features, d_model)
    x = layers.Reshape((n_features, 1))(inputs)
    x = layers.Dense(d_model, name='feature_embed')(x)

    # Prepend [CLS] token → (batch, n_features+1, d_model)
    x = CLSToken(d_model, name='cls_token')(x)

    # Learnable positional embedding
    positions = tf.range(start=0, limit=n_features + 1)
    pos_embed = layers.Embedding(input_dim=n_features + 1,
                                 output_dim=d_model,
                                 name='pos_embed')(positions)
    x = x + pos_embed

    for i in range(num_blocks):
        x = TransformerEncoderBlock(
            d_model, nhead, ff_dim, dropout, name=f'enc_block_{i}')(x)

    # Extract [CLS] token output
    cls_out = x[:, 0, :]                     # (batch, d_model)
    cls_out = layers.LayerNormalization()(cls_out)
    cls_out = layers.Dense(64, activation='relu')(cls_out)
    cls_out = layers.Dropout(dropout)(cls_out)
    outputs = layers.Dense(1, name='output')(cls_out)
    return Model(inputs, outputs, name='Transformer')


tfm_model = build_transformer(N_FEATURES)
tfm_model.summary()

In [ ]:
tfm_history = fit_model(tfm_model, 'Transformer')
tfm_preds, tfm_trues, tfm_metrics = evaluate_model(tfm_model, 'Transformer')

---
## 9. Comparative Results

In [ ]:
results = pd.DataFrame({
    'Model':    ['MLP', 'CNN-1D', 'LSTM', 'Transformer'],
    'RMSE':     [mlp_metrics['RMSE'],  cnn_metrics['RMSE'],
                 lstm_metrics['RMSE'], tfm_metrics['RMSE']],
    'MAE':      [mlp_metrics['MAE'],   cnn_metrics['MAE'],
                 lstm_metrics['MAE'],  tfm_metrics['MAE']],
    'R²':       [mlp_metrics['R2'],    cnn_metrics['R2'],
                 lstm_metrics['R2'],   tfm_metrics['R2']],
    'MAPE (%)': [mlp_metrics['MAPE'],  cnn_metrics['MAPE'],
                 lstm_metrics['MAPE'], tfm_metrics['MAPE']],
})

results = results.sort_values('RMSE').reset_index(drop=True)
results.style \
    .background_gradient(subset=['RMSE', 'MAE', 'MAPE (%)'], cmap='RdYlGn_r') \
    .background_gradient(subset=['R²'], cmap='RdYlGn') \
    .format({'RMSE': '{:.4f}', 'MAE': '{:.4f}', 'R²': '{:.4f}', 'MAPE (%)': '{:.2f}'})

In [ ]:
# ── Bar charts ────────────────────────────────────────────────────────────────
models    = results['Model'].tolist()
color_map = {'MLP': '#2196F3', 'CNN-1D': '#FF9800',
             'LSTM': '#4CAF50', 'Transformer': '#9C27B0'}
bar_colors = [color_map[m] for m in models]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (metric, lower) in zip(axes, [('RMSE', True), ('MAE', True),
                                       ('R²', False), ('MAPE (%)', True)]):
    vals = results[metric].tolist()
    bars = ax.bar(models, vals, color=bar_colors, edgecolor='white', linewidth=1.2)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticklabels(models, rotation=15)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('↓ better' if lower else '↑ better', fontsize=10, color='gray')

plt.suptitle('Model Comparison — Oil Recovery Factor (Test Set)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────────
histories = {
    'MLP':         mlp_history,
    'CNN-1D':      cnn_history,
    'LSTM':        lstm_history,
    'Transformer': tfm_history,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    c = color_map[name]
    axes[0].plot(hist.history['loss'],     label=name, color=c)
    axes[1].plot(hist.history['val_loss'], label=name, color=c)

for ax, title in zip(axes, ['Training Loss (MSE)', 'Validation Loss (MSE)']):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (scaled space)')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Actual vs Predicted ───────────────────────────────────────────────────────
all_preds = {
    'MLP':         (mlp_trues,  mlp_preds),
    'CNN-1D':      (cnn_trues,  cnn_preds),
    'LSTM':        (lstm_trues, lstm_preds),
    'Transformer': (tfm_trues,  tfm_preds),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, (name, (trues, preds)) in zip(axes.ravel(), all_preds.items()):
    r2   = r2_score(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    ax.scatter(trues, preds, alpha=0.4, s=15, color=color_map[name], edgecolors='none')
    lo, hi = min(trues.min(), preds.min()), max(trues.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, label='Perfect fit')
    ax.set_title(f'{name}  (R²={r2:.4f}, RMSE={rmse:.4f})', fontsize=11)
    ax.set_xlabel('Actual Oil Recovery (%)')
    ax.set_ylabel('Predicted Oil Recovery (%)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

plt.suptitle('Actual vs. Predicted — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residual distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, (trues, preds)) in zip(axes.ravel(), all_preds.items()):
    residuals = trues.ravel() - preds.ravel()
    ax.hist(residuals, bins=50, color=color_map[name], edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name} — Residuals', fontsize=11)
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.25)

plt.suptitle('Residual Distributions — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('residuals.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Feature Importance via Permutation

Shuffle one feature at a time, measure the increase in test RMSE.  
Larger increase = more important. Applied to the best-performing model.

In [ ]:
def permutation_importance(model, X_s, y_inv, y_scaler,
                            feature_names, n_repeats=5):
    base_pred_inv = y_scaler.inverse_transform(model.predict(X_s, verbose=0))
    base_rmse     = np.sqrt(mean_squared_error(y_inv, base_pred_inv))

    imp = np.zeros((len(feature_names), n_repeats))
    for i in range(len(feature_names)):
        for r in range(n_repeats):
            Xp = X_s.copy()
            np.random.shuffle(Xp[:, i])
            pred_inv  = y_scaler.inverse_transform(model.predict(Xp, verbose=0))
            perm_rmse = np.sqrt(mean_squared_error(y_inv, pred_inv))
            imp[i, r] = perm_rmse - base_rmse
    return imp.mean(axis=1), imp.std(axis=1)


best_name  = results.iloc[0]['Model']
model_map  = {'MLP': mlp_model, 'CNN-1D': cnn_model,
              'LSTM': lstm_model, 'Transformer': tfm_model}
best_model = model_map[best_name]
print(f'Computing permutation importance for best model: {best_name}')

imp_mean, imp_std = permutation_importance(
    best_model, X_test_s, y_scaler.inverse_transform(y_test_s),
    y_scaler, FEATURES, n_repeats=5)

sorted_idx = np.argsort(imp_mean)[::-1]

plt.figure(figsize=(10, 6))
plt.barh([FEATURES[i] for i in sorted_idx], imp_mean[sorted_idx],
         xerr=imp_std[sorted_idx],
         color='steelblue', alpha=0.85, edgecolor='white')
plt.xlabel('Mean RMSE increase (permutation importance)')
plt.title(f'Feature Importance — {best_name} (Best Model)', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Final Summary

In [ ]:
print('=' * 60)
print('   FINAL COMPARISON — OIL RECOVERY FACTOR PREDICTION')
print('=' * 60)
print(results.to_string(index=False))
print()
print('Best model by RMSE :', results.iloc[0]['Model'])
print('Best model by R²   :', results.loc[results['R²'].idxmax(), 'Model'])

---
## Discussion

| Model | Strength | Weakness |
|---|---|---|
| **MLP** | Fast, simple, strong baseline on tabular data | No local/sequential feature correlations |
| **CNN-1D** | Efficient local feature interactions via convolutions | Feature order is arbitrary; no long-range dependencies |
| **LSTM** | Gated memory captures ordered feature interactions | Slower training; less parallelisable than CNN/Transformer |
| **Transformer** | Full pairwise self-attention; most expressive | Needs more data/tuning to beat simpler models; higher memory |

**Key takeaway:** All four models are trained with identical hyper-parameters and callbacks for a fair comparison.  
The best model is identified above; R² > 0.9 indicates the proxy is suitable for fast reservoir screening.